In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score


In [9]:
# Load reduced dataset
# -----------------------------
df = pd.read_csv("../data/heart_disease_reduced.csv")

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [10]:
# Define search spaces
# -----------------------------
search_spaces = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, solver="liblinear"),
        {"C": [0.01, 0.1, 1, 10, 100], "penalty": ["l1", "l2"]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"max_depth": [2, 4, 6, 8, 10, None], "min_samples_split": [2, 5, 10]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [50, 100, 200, 300],
            "max_depth": [4, 6, 8, None],
            "min_samples_split": [2, 5, 10],
            "criterion": ["gini", "entropy"]
        }
    ),
    "SVM": (
        SVC(probability=True, random_state=42),
        {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"], "gamma": ["scale", "auto"]}
    )
}


In [11]:
# -----------------------------
# Run tuning for each model
# -----------------------------
best_models = {}
scores = {}

for name, (model, params) in search_spaces.items():
    if name == "Random Forest":
        search = RandomizedSearchCV(model, params, n_iter=10, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1)
    else:
        search = GridSearchCV(model, params, cv=5, scoring="roc_auc", n_jobs=-1)
    
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_models[name] = best_model
    
    y_proba = best_model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    scores[name] = auc
    
    print(f"\n✅ {name} Best Params: {search.best_params_}")
    print("ROC AUC:", auc)
    print(classification_report(y_test, best_model.predict(X_test)))



✅ Logistic Regression Best Params: {'C': 0.1, 'penalty': 'l2'}
ROC AUC: 0.9610389610389611
              precision    recall  f1-score   support

           0       0.90      0.82      0.86        33
           1       0.81      0.89      0.85        28

    accuracy                           0.85        61
   macro avg       0.85      0.86      0.85        61
weighted avg       0.86      0.85      0.85        61


✅ Decision Tree Best Params: {'max_depth': 10, 'min_samples_split': 10}
ROC AUC: 0.7857142857142858
              precision    recall  f1-score   support

           0       0.81      0.79      0.80        33
           1       0.76      0.79      0.77        28

    accuracy                           0.79        61
   macro avg       0.79      0.79      0.79        61
weighted avg       0.79      0.79      0.79        61


✅ Random Forest Best Params: {'n_estimators': 100, 'min_samples_split': 10, 'max_depth': 8, 'criterion': 'gini'}
ROC AUC: 0.9491341991341992
           

In [12]:
# Auto-select best model
# -----------------------------
best_model_name = max(scores, key=scores.get)
best_model = best_models[best_model_name]

print("\n🎯 Best Model Selected:", best_model_name, "with ROC AUC =", scores[best_model_name])

# Save best model name for Notebook 7
with open("../models/best_model_name.txt", "w") as f:
    f.write(best_model_name)


🎯 Best Model Selected: SVM with ROC AUC = 0.9653679653679654
